# 🔄 RESUME TRAINING — Sesi Lanjutan (Batas Waktu: 5 Jam)
> Notebook ini melanjutkan training dari checkpoint terakhir.
> Pastikan folder checkpoint sesi sebelumnya sudah tersedia (mount sebagai dataset Kaggle).
> Notebook ini bisa dijalankan berulang kali hingga training selesai.

## Install Dependencies

In [ ]:
!pip install evaluate jiwer -q

## ⚙️ Konfigurasi Path Checkpoint
> **Edit cell ini** sesuai lokasi checkpoint dari sesi sebelumnya.

In [ ]:
import os

# =================================================================
# 🔧 EDIT SESUAI KEBUTUHAN
# Lokasi folder checkpoint dari sesi sebelumnya.
# Jika di-mount sebagai Kaggle Dataset, biasanya ada di /kaggle/input/
# =================================================================
CHECKPOINT_BASE = "/kaggle/input/whisper-balinese-checkpoint/whisper-balinese"

# Otomatis cari checkpoint terbaru
if os.path.exists(CHECKPOINT_BASE):
    checkpoints = sorted(
        [d for d in os.listdir(CHECKPOINT_BASE) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[1])
    )
    if checkpoints:
        RESUME_FROM = os.path.join(CHECKPOINT_BASE, checkpoints[-1])
        print(f"✅ Checkpoint ditemukan: {RESUME_FROM}")
    else:
        # Tidak ada subfolder checkpoint — gunakan base langsung (save_model output)
        RESUME_FROM = CHECKPOINT_BASE
        print(f"✅ Menggunakan base checkpoint: {RESUME_FROM}")
else:
    raise FileNotFoundError(
        f"❌ Folder checkpoint tidak ditemukan: {CHECKPOINT_BASE}\n"
        "   Pastikan dataset checkpoint sudah di-mount dengan benar."
    )

## Get Dataset

In [ ]:
!git clone https://huggingface.co/datasets/Sparkplugx1904/Balinese-Common-Voice/ temp
!mv temp/* ./
!rm -rf temp

## Import Library

In [ ]:
from datasets import load_dataset, Audio, Features, Value, Dataset
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
import torch
import pandas as pd
import librosa

## Data Loading and Processing

In [ ]:
import pandas as pd
from datasets import Dataset, Value
from sklearn.model_selection import train_test_split

metadata_df = pd.read_csv("metadata.tsv", sep='\t')
metadata_df = metadata_df[["path", "balinese"]].rename(columns={"balinese": "sentence"})

if not metadata_df["path"].str.startswith("clips/").all():
    metadata_df["path"] = "clips/" + metadata_df["path"].astype(str)

metadata_df = metadata_df.dropna(subset=["path", "sentence"]).reset_index(drop=True)

train_df, test_df = train_test_split(metadata_df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Total data: {len(metadata_df)}")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset  = Dataset.from_pandas(test_df,  preserve_index=False)
train_dataset = train_dataset.cast_column("path", Value("string"))
test_dataset  = test_dataset.cast_column("path",  Value("string"))

print(train_dataset)

## Load Whisper Processor

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")
tokenizer  = WhisperTokenizer.from_pretrained("openai/whisper-medium", language="indonesian", task="transcribe")
processor  = WhisperProcessor.from_pretrained("openai/whisper-medium", language="indonesian", task="transcribe")

def prepare_dataset(batch):
    batch["labels"] = processor.tokenizer(batch["sentence"], truncation=True).input_ids
    return batch

print("Melakukan tokenisasi dataset...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=["sentence"])
test_dataset  = test_dataset.map(prepare_dataset,  remove_columns=["sentence"])
print(f"Kolom: {train_dataset.column_names}")

## Load Model dari Checkpoint

In [ ]:
from transformers import WhisperForConditionalGeneration

# Load model dari checkpoint sesi sebelumnya (bukan dari openai/whisper-medium)
model = WhisperForConditionalGeneration.from_pretrained(RESUME_FROM)
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []

print(f"✅ Model di-load dari checkpoint: {RESUME_FROM}")

## Collator

In [ ]:
import librosa
import numpy as np
import torch

class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor
        self.sr = 16000
        self.pad = processor.tokenizer.pad_token_id

    def __call__(self, features):
        input_features = [{
            "input_features": self.processor.feature_extractor(
                librosa.load(feature["path"], sr=self.sr)[0],
                sampling_rate=self.sr
            ).input_features[0]
        } for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        labels = [torch.tensor(feature["labels"]) for feature in features]
        labels_padded = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=self.pad
        )
        batch["labels"] = labels_padded
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

## Metric (WER)

In [ ]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("Metric WER siap.")

## 🔄 Resume Training (Sesi Lanjutan — Maks 5 Jam)

In [ ]:
import os
import gc
import time
import shutil
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback

torch.cuda.empty_cache()
gc.collect()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
model.config.use_cache = False

# ============================================================
# ⏱️ CALLBACK: Stop otomatis 15 menit sebelum batas 5 jam
# ============================================================
MAX_HOURS  = 5
SAFETY_MIN = 15

class TimeLimitCallback(TrainerCallback):
    def __init__(self, max_hours, safety_minutes):
        self.deadline = time.time() + (max_hours * 3600) - (safety_minutes * 60)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 20 == 0:
            torch.cuda.empty_cache()
            gc.collect()
        if time.time() >= self.deadline:
            print(f"\n⏰ Batas waktu {MAX_HOURS} jam hampir habis. Menghentikan training...")
            print(f"   Checkpoint terakhir akan disimpan di: ./whisper-balinese")
            control.should_training_stop = True
        return control

    def on_train_end(self, args, state, control, **kwargs):
        sisa = max(0, self.deadline - time.time())
        print(f"\n✅ Training dihentikan pada step {state.global_step}, epoch {state.epoch:.2f}")
        print(f"   Sisa waktu (buffer): {sisa/60:.1f} menit")
        print(f"   ➡️  Jalankan lagi notebook ini untuk melanjutkan sesi berikutnya")

# Output ke working dir (bukan read-only input)
OUTPUT_DIR = "./whisper-balinese"

# Salin checkpoint ke working dir agar bisa ditulis trainer
if not os.path.exists(OUTPUT_DIR):
    print(f"📋 Menyalin checkpoint ke working dir...")
    shutil.copytree(CHECKPOINT_BASE, OUTPUT_DIR)
    print(f"✅ Checkpoint tersalin ke: {OUTPUT_DIR}")
else:
    print(f"✅ Output dir sudah ada: {OUTPUT_DIR}")

# Cari checkpoint terbaru di working dir
local_checkpoints = sorted(
    [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
if local_checkpoints:
    LOCAL_RESUME = os.path.join(OUTPUT_DIR, local_checkpoints[-1])
    print(f"🔄 Resume dari: {LOCAL_RESUME}")
else:
    LOCAL_RESUME = True   # Biarkan trainer cari sendiri
    print(f"🔄 Resume dari checkpoint terakhir di: {OUTPUT_DIR}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,

    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=30,
    lr_scheduler_type="cosine",
    weight_decay=0.01,

    fp16=True,
    gradient_checkpointing=True,

    predict_with_generate=True,
    generation_max_length=225,
    eval_accumulation_steps=1,

    eval_strategy="steps",
    logging_steps=5,
    save_strategy="steps",
    save_steps=200,
    eval_steps=200,
    report_to=["tensorboard"],

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[TimeLimitCallback(MAX_HOURS, SAFETY_MIN)]
)

print(f"\n🔄 Melanjutkan Training...")
print(f"⏱️  Akan berjalan maks {MAX_HOURS} jam (stop {SAFETY_MIN} menit sebelum batas)")
print()

# resume_from_checkpoint otomatis lanjut dari step terakhir
trainer.train(resume_from_checkpoint=LOCAL_RESUME)

## 💾 Simpan Checkpoint Sesi Ini

In [ ]:
import os

trainer.save_model("./whisper-balinese")
processor.save_pretrained("./whisper-balinese")
tokenizer.save_pretrained("./whisper-balinese")

checkpoints = sorted([d for d in os.listdir("./whisper-balinese") if d.startswith("checkpoint-")])
print("✅ Checkpoint tersimpan:")
for ck in checkpoints:
    print(f"   📁 ./whisper-balinese/{ck}")
print()
print("➡️  Upload ulang folder './whisper-balinese' ke Kaggle Dataset")
print("   lalu jalankan lagi notebook ini untuk sesi berikutnya")